In [137]:
import re
import json
import html
import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

scraper = cloudscraper.create_scraper()

In [138]:
# variables

state = 'mg'

In [139]:
state = 'mg'
content = scraper.get(f"https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/estado-{state}?lis=home_body_search_bar_2020").text

In [ ]:
linhas_anuncios = []
for page in range(1, 3):
    content = scraper.get(f"https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/estado-{state}?lis=home_body_search_bar_2020&o={page}").text
    soup = BeautifulSoup(content, "html.parser")

    for card in soup.select("section.olx-adcard"):
        link = card.select_one('a[data-testid="adcard-link"]')
        if not link:
            continue

        url = urljoin("https://www.olx.com.br", link.get("href", ""))
        imagem = card.select_one(".olx-adcard__media img")
        detalhes_card = [
            item.get("aria-label", item.get_text(" ", strip=True))
            for item in card.select(".olx-adcard__detail")
        ]

        linha = {
            "url": url,
            "titulo_card": link.get_text(" ", strip=True),
            "preco_card": (
                float(
                    card.select_one(".olx-adcard__price")
                    .get_text(" ", strip=True)
                    .replace("R$", "")
                    .replace(".", "")
                    .replace(",", ".")
                    .strip()
                )
                if card.select_one(".olx-adcard__price")
                else None
            ),
            "local_card": (
                card.select_one(".olx-adcard__location").get_text(" ", strip=True)
                if card.select_one(".olx-adcard__location") else None
            ),
            "data_card": (
                card.select_one(".olx-adcard__date").get_text(" ", strip=True)
                if card.select_one(".olx-adcard__date") else None
            ),
            "detalhes_card": " | ".join(detalhes_card),
            "imagem_card": imagem.get("src") if imagem else None,
            "erro_coleta": None,
        }

        try:
            resposta = scraper.get(url)
            resposta.raise_for_status()
            html_anuncio = resposta.text
            pagina = extrair_datalayer(html_anuncio)

            # Campos gerais do anúncio: título, preço, descrição, localização,
            # marca, modelo, ano, quilometragem, combustível, câmbio etc.
            ad_detail = pagina.get("adDetail") or {}
            if isinstance(ad_detail, dict):
                # O adDate de adDetail vem incorreto (data em 1970); guardamos
                # o valor original à parte e usamos o timestamp de detail abaixo.
                linha.update({f"anuncio_{chave}": valor_tabelavel(valor)
                            for chave, valor in ad_detail.items()
                            if chave != "adDate"})

            # Use o timestamp Unix de detail.adDate e transforme diretamente
            # a coluna anuncio_adDate em datetime local de São Paulo.
            detail = pagina.get("detail") or {}
            if isinstance(detail, dict):
                timestamp_anuncio = detail.get("adDate")
                try:
                    if timestamp_anuncio is not None:
                        timestamp_anuncio = float(timestamp_anuncio)
                        unidade = "ms" if abs(timestamp_anuncio) >= 1e12 else "s"
                        linha["anuncio_adDate"] = pd.to_datetime(
                            timestamp_anuncio, unit=unidade, utc=True
                        ).tz_convert("America/Sao_Paulo")
                    else:
                        linha["anuncio_adDate"] = pd.NaT
                except (TypeError, ValueError, OverflowError):
                    linha["anuncio_adDate"] = pd.NaT

                referencia = detail.get("abuyPriceRef") or {}
                if isinstance(referencia, dict):
                    linha.update({f"referencia_olx_{chave}": valor
                                for chave, valor in referencia.items()})

            # Atributos estruturados do carro (ano, quilometragem, cor,
            # opcionais, documentação e outros que estiverem disponíveis).
            propriedades = pagina.get("adProperties") or []
            if isinstance(propriedades, list):
                for propriedade in propriedades:
                    if not isinstance(propriedade, dict):
                        continue
                    nome = propriedade.get("name") or propriedade.get("label")
                    if nome:
                        linha[f"caracteristica_{nome}"] = valor_tabelavel(
                            propriedade.get("value")
                        )

            # O valor FIPE fica em outro bloco de estado embutido no HTML.
            linha["preco_fipe"] = extrair_preco_fipe(html_anuncio)

        except Exception as erro:
            linha["erro_coleta"] = str(erro)
            linha["preco_fipe"] = None

        linhas_anuncios.append(linha)

df_anuncios = pd.DataFrame(linhas_anuncios).drop_duplicates(subset="url")

# Campos inteiros para colunas bigint: IDs, ano, quilometragem, DDD,
# final da placa, mês de referência e quantidade de anúncios.
colunas_bigint = [
    "anuncio_adId", "anuncio_listId", "anuncio_mainCategoryID",
    "anuncio_subCategoryID", "anuncio_ddd", "anuncio_regdate",
    "anuncio_mileage", "anuncio_end_tag", "caracteristica_regdate",
    "caracteristica_mileage", "caracteristica_end_tag",
    "referencia_olx_year_month_ref", "referencia_olx_vehicle_count",
]

# Valores monetários aceitam parte decimal. No Supabase, configure esses
# campos como numeric (e não bigint). Motorpower fica como texto porque pode
# vir como faixa (por exemplo, "2.0 - 2.9").
colunas_numeric = [
    "preco_card", "anuncio_price", "preco_fipe",
    "referencia_olx_price_min", "referencia_olx_price_p25",
    "referencia_olx_price_p33", "referencia_olx_price_p50",
    "referencia_olx_price_p66", "referencia_olx_price_p75",
    "referencia_olx_price_max", "referencia_olx_price_stddev",
]

for coluna in colunas_bigint:
    if coluna in df_anuncios.columns:
        valores = pd.to_numeric(df_anuncios[coluna], errors="coerce")
        preenchidos = df_anuncios[coluna].notna() & df_anuncios[coluna].astype(str).str.strip().ne("")
        invalidos = preenchidos & valores.isna()
        fracionarios = valores.notna() & valores.mod(1).ne(0)
        if invalidos.any() or fracionarios.any():
            raise ValueError(
                f"A coluna {coluna} deveria conter apenas inteiros; "
                f"valores inválidos: {df_anuncios.loc[invalidos | fracionarios, coluna].head().tolist()}"
            )
        df_anuncios[coluna] = valores.astype("Int64")

for coluna in colunas_numeric:
    if coluna in df_anuncios.columns:
        valores = pd.to_numeric(df_anuncios[coluna], errors="coerce")
        preenchidos = df_anuncios[coluna].notna() & df_anuncios[coluna].astype(str).str.strip().ne("")
        if (preenchidos & valores.isna()).any():
            raise ValueError(
                f"A coluna {coluna} contém valores que não são numéricos: "
                f"{df_anuncios.loc[preenchidos & valores.isna(), coluna].head().tolist()}"
            )
        df_anuncios[coluna] = valores.astype("Float64")

df_anuncios.to_csv("anuncios_detalhados.csv", index=False, encoding="utf-8-sig")

In [141]:
df_anuncios

,url,titulo_card,preco_card,local_card,data_card,detalhes_card,imagem_card,erro_coleta,anuncio_adId,anuncio_listId,...,caracteristica_conservation_and_warranty,caracteristica_indexed_car_steering,caracteristica_on_autos_fair,caracteristica_condition_report_date,caracteristica_condition_report_detail,caracteristica_condition_report_inspector,caracteristica_condition_report_status,caracteristica_condition_report_visible,caracteristica_store_id,preco_fipe
0,https://mg.olx.com.br/belo-horizonte-e-regiao/...,Volkswagen T-Cross Highline 1.4 TSI Flex 16V 5...,152990.0,"Belo Horizonte, Padre Eustáquio","Hoje, 13:02",47721 quilômetros rodados | Cor Prata | Motor ...,https://img.olx.com.br/thumbs700x500/57/575618...,NaN,1.808515e+09,1.534516e+09,...,Chave reserva,Elétrica,Sim,2026-10-01T13:46:12.938Z,A vistoriadora integrada retornou que não exis...,Visão Total,UNAVAILABLE,true,Localiza,159834.0
1,https://mg.olx.com.br/regiao-de-divinopolis/au...,Volkswagen Saveiro Extreme 1.6 Flex 16V CD 2024,99900.0,Carmópolis de Minas,"Hoje, 13:02",24000 quilômetros rodados | Cor Cinza | Motor ...,https://img.olx.com.br/thumbs700x500/16/169699...,NaN,1.800035e+09,1.526034e+09,...,"Único dono, Com manual, Chave reserva",Hidráulica,Não,2026-09-11T09:58:08.219Z,A vistoriadora integrada retornou que não exis...,Super Visão,UNAVAILABLE,true,NaN,98366.0
2,https://mg.olx.com.br/belo-horizonte-e-regiao/...,Ford Fusion Titanium 2.0 Gtdi Eco. FWD Aut. 2014,66000.0,"Belo Horizonte, Jardim Guanabara","Hoje, 13:02",199000 quilômetros rodados | Cor Branco | Moto...,https://img.olx.com.br/thumbs700x500/21/217695...,503 Server Error: Service Unavailable for url:...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://mg.olx.com.br/belo-horizonte-e-regiao/...,Volkswagen Taos Highline 1.4 250 TSI Flex AUT ...,150690.0,"Belo Horizonte, Liberdade","Hoje, 13:02",49711 quilômetros rodados | Cor Preto | Motor ...,https://img.olx.com.br/thumbs700x500/65/651685...,NaN,1.802959e+09,1.528958e+09,...,Chave reserva,Elétrica,Não,2026-09-30T04:02:09.558Z,A vistoriadora integrada retornou que não exis...,Super Visão,UNAVAILABLE,true,Localiza,152895.0
4,https://mg.olx.com.br/regiao-de-uberlandia-e-u...,Fiat Mobi Like 1.0 Fire Flex 5P. 2025,62990.0,"Uberlândia, Santa Mônica","Hoje, 13:02",29237 quilômetros rodados | Cor Branco | Motor...,https://img.olx.com.br/thumbs700x500/28/282683...,NaN,1.808516e+09,1.534517e+09,...,"Único dono, Com manual, Chave reserva",Elétrica,Sim,2026-09-25T11:30:35.834Z,A vistoriadora integrada retornou que não exis...,Super Visão,UNAVAILABLE,true,Localiza,61634.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,https://mg.olx.com.br/belo-horizonte-e-regiao/...,Volkswagen Voyage 1.6 MSI Flex 8V 4P 2018,51900.0,"Belo Horizonte, Venda Nova","Hoje, 12:50",139108 quilômetros rodados | Cor Prata | Motor...,https://img.olx.com.br/thumbs700x500/62/629678...,NaN,1.814845e+09,1.540847e+09,...,0,Mecânica,Não,2026-10-02T15:52:56.610Z,A vistoriadora integrada retornou que não exis...,Visão Total,UNAVAILABLE,true,NaN,NaN
96,https://mg.olx.com.br/belo-horizonte-e-regiao/...,Chevrolet Onix Plus LT 1.0 12V Flex Mec. 4P 2025,65000.0,"Belo Horizonte, Padre Eustáquio","Hoje, 12:50",131000 quilômetros rodados | Cor Cinza | Motor...,https://img.olx.com.br/thumbs700x500/95/950671...,NaN,1.813069e+09,1.539070e+09,...,"Único dono, Com manual, Chave reserva, Com gar...",Elétrica,Não,NaN,NaN,NaN,NaN,NaN,NaN,77031.0
97,https://mg.olx.com.br/belo-horizonte-e-regiao/...,Fiat Toro Ranch 2.0 16V 4X4 Diesel Aut. 2025,159900.0,"Belo Horizonte, Barreiro","Hoje, 12:49",15271 quilômetros rodados | Cor Prata | Motor ...,https://img.olx.com.br/thumbs700x500/32/327627...,NaN,1.810725e+09,1.536726e+09,...,0,Elétrica,Não,2026-09-18T17:22:43.222Z,Não encontramos o laudo. Tentaremos novamente ...,Super Visão,ERROR,true,NaN,160231.0
98,https://mg.olx.com.br/belo-horizonte-e-regiao/...,Volkswagen Voyage 1.6 MSI Flex 8V 4P 2017,45900.0,"Belo Horizonte,

### Sincronização SCD2 com Supabase

Antes de executar a célula seguinte, rode `supabase_scd2.sql` no SQL Editor. A sincronização usa `dt_loadtime`, `flag_current` e `version` para manter o histórico. Configure `SUPABASE_URL` e `SUPABASE_SERVICE_ROLE_KEY` em um arquivo local `.env`, nunca no notebook ou no repositório.

A chave service role deve ser usada somente em ambiente confiável. O schema `bronze` precisa estar exposto na API Supabase para acesso pelo cliente Python.

In [ ]:
# Instale uma vez, se necessário: %pip install supabase python-dotenv
from supabase_sync import sync_ads_dataframe

resultado_scd2 = sync_ads_dataframe(df_anuncios)
print("Resultado SCD2:", resultado_scd2)
